In [0]:
from pyspark.sql import functions as F
import re

# ---------- Configuration ----------
CATALOG = "oil_gas_demo"
SOURCE_PATH = "/Volumes/oil_gas_demo/raw/raw_volume/your_file.csv"  # change file name
TARGET_TABLE = f"{CATALOG}.bronze.supply_chain_transactions"

# ---------- Read CSV as-is ----------
# All original CSV columns remain strings in Bronze.
bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(SOURCE_PATH)
)

# Standard Delta-safe column naming
def to_snake_case(column_name):
    return re.sub(r"[^a-z0-9]+", "_", column_name.strip().lower()).strip("_")

bronze_df = bronze_df.toDF(*[to_snake_case(c) for c in bronze_df.columns])

# ---------- Add date and audit columns ----------
bronze_df = (
    bronze_df
    .withColumn(
        "txn_date",
        F.coalesce(
            F.to_date(F.col("transaction_date"), "yyyy-MM-dd"),
            F.to_date(F.col("transaction_date"), "dd-MM-yyyy"),
            F.to_date(F.col("transaction_date"), "MM/dd/yyyy"),
            F.to_date(F.col("transaction_date"), "dd/MM/yyyy"),
            F.to_date(F.col("transaction_date"), "yyyy-MM-dd HH:mm:ss"),
            F.to_date(F.col("transaction_date"), "yyyy-MM-dd'T'HH:mm:ss")
        )
    )
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name())
)

# ---------- Write Delta Bronze table ----------
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

# ---------- Validation ----------
print(f"Bronze table created: {TARGET_TABLE}")
print(f"Row count: {spark.table(TARGET_TABLE).count()}")

display(spark.table(TARGET_TABLE).limit(10))

display(
    spark.table(TARGET_TABLE).select(
        F.count("*").alias("total_rows"),
        F.sum(F.col("txn_date").isNull().cast("int")).alias("null_txn_dates"),
        F.countDistinct("transaction_id").alias("distinct_transaction_ids")
    )
)

spark.table(TARGET_TABLE).printSchema()